In [45]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import time

In [12]:
spark = SparkSession.builder \
    .appName("Analyze_Ecommerce_Data") \
    .master("spark://spark-master:7077") \
    .config("spark.ui.port", "4041") \
    .config("spark.executor.instances","2") \
    .config("spark.executor.cores","1") \
    .config("spark.executor.memory","1g") \
    .getOrCreate()

In [13]:
df = spark.read \
    .format("parquet") \
    .option("header","true") \
    .load("/opt/spark/work-dir/data/ecommerce_data")

In [15]:
df = df.withColumn(
    "order_month",
    date_format(col("order_date"), "yyyy-MM")
)

In [16]:
df.show()

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-------------+-----------+
|order_id|customer_id|product_id|   category| price|quantity|order_status|order_date|payment_method|country|total_revenue|order_month|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-------------+-----------+
|       1|         52|       380|       Home|643.03|       3|     success|2026-03-12| Bank Transfer|     US|      1929.09|    2026-03|
|      10|        284|       261|     Sports|227.13|       1|      failed|2024-01-13| Bank Transfer| Canada|       227.13|    2024-01|
|     100|         29|       136|     Beauty|792.78|       6|     success|2024-08-09|        PayPal|     US|      4756.68|    2024-08|
|    1000|       1435|       319|     Sports|508.21|       4|     success|2023-07-12|        PayPal| France|      2032.84|    2023-07|
|    1001|         27|       342|     Sports|150.42|   

In [18]:
revenue_by_month = df.groupBy("order_month").agg(
    round(sum("total_revenue"),2).alias("revenue_by_month")
)

In [19]:
revenue_by_month.show()

[Stage 5:>                                                          (0 + 1) / 1]

+-----------+----------------+
|order_month|revenue_by_month|
+-----------+----------------+
|    2026-05|       277555.38|
|    2024-09|       315740.74|
|    2023-08|       297439.42|
|    2024-02|       280751.15|
|    2023-12|       304085.33|
|    2025-04|       301192.65|
|    2026-06|         29136.1|
|    2023-11|       295491.19|
|    2025-01|       280688.37|
|    2025-08|       304140.45|
|    2023-07|        259887.5|
|    2024-08|       277119.86|
|    2023-03|       320600.16|
|    2023-10|       348501.75|
|    2026-04|       259702.26|
|    2024-06|       291305.74|
|    2023-02|        81061.86|
|    2023-09|       343687.48|
|    2023-04|       242781.12|
|    2023-05|       330170.91|
+-----------+----------------+
only showing top 20 rows



In [20]:
revenue_by_category = df.groupBy('category').agg(
    round(sum("total_revenue"),2).alias('revenue_by_category')
).show()

+-----------+-------------------+
|   category|revenue_by_category|
+-----------+-------------------+
|       Home|         2063246.14|
|    Fashion|         2002629.13|
|     Sports|         2130138.58|
|Electronics|          1848587.8|
|      Books|         2004399.57|
|     Beauty|         2017236.28|
+-----------+-------------------+



In [21]:
revenue_by_country = df.groupBy("country").agg(
    round(sum("total_revenue"),2).alias("revenue_by_country")
).show()

+-------+------------------+
|country|revenue_by_country|
+-------+------------------+
|Germany|        1691944.78|
| France|        1879577.22|
|     US|        1748998.53|
|     UK|        1659707.12|
| Canada|        1566020.29|
|  Japan|        1765145.31|
|Vietnam|        1754844.25|
+-------+------------------+



In [22]:
top_customer = df.groupBy('customer_id').agg(
    round(sum("total_revenue"),2).alias("customer_spending")
).orderBy(col("customer_spending").desc()).show()

+-----------+-----------------+
|customer_id|customer_spending|
+-----------+-----------------+
|       1825|         32539.06|
|       1816|          30388.4|
|        711|          30184.5|
|       1309|         28968.65|
|       1926|         28309.25|
|        213|         27728.48|
|        127|         25780.71|
|       1752|         24550.06|
|       1211|         24508.28|
|        172|         24409.68|
|        910|         24335.67|
|       1359|         24151.12|
|         90|         23622.75|
|       1982|         23614.98|
|         32|         23221.34|
|       1072|         23050.99|
|          3|         23045.37|
|        754|         22665.73|
|        970|         22363.95|
|       1804|         22053.36|
+-----------+-----------------+
only showing top 20 rows



In [24]:
df.select(
    round(
        count(when(col('order_status') == 'failed', True)) / count("*") * 100, 2
    ).alias("failed_order_percentage")
).show()


+-----------------------+
|failed_order_percentage|
+-----------------------+
|                  32.58|
+-----------------------+



In [33]:
order_frequency = df.groupBy('customer_id').agg(count("*").alias("customer_frequency_order")).orderBy(col("customer_frequency_order").desc())
order_frequency.show()

+-----------+------------------------+
|customer_id|customer_frequency_order|
+-----------+------------------------+
|        711|                      10|
|        213|                      10|
|       1309|                       9|
|       1926|                       8|
|         90|                       8|
|       1400|                       8|
|       1989|                       7|
|        970|                       7|
|        220|                       7|
|        874|                       7|
|       1982|                       7|
|        448|                       7|
|       1816|                       7|
|        910|                       7|
|       1186|                       7|
|        643|                       7|
|       1072|                       7|
|        127|                       7|
|       1503|                       7|
|       1594|                       6|
+-----------+------------------------+
only showing top 20 rows



In [41]:
monthly_sales = df.groupBy(col("order_month").alias("month"),"category").agg(
    round(sum("total_revenue"),2).alias("total_revenue")
)

In [42]:
monthly_sales.show()

+-------+-----------+-------------+
|  month|   category|total_revenue|
+-------+-----------+-------------+
|2025-02|     Sports|     58968.48|
|2023-10|       Home|     51513.49|
|2025-02|      Books|     48856.47|
|2024-01|       Home|     42929.71|
|2023-08|       Home|     51438.28|
|2023-07|      Books|      52263.6|
|2023-09|     Beauty|     47867.02|
|2024-05|     Sports|     76032.66|
|2024-02|     Beauty|     34188.08|
|2024-09|       Home|     62011.97|
|2023-12|Electronics|     29078.57|
|2026-02|    Fashion|     40562.77|
|2026-04|     Sports|     55462.97|
|2025-10|       Home|     51342.92|
|2026-04|       Home|     39487.66|
|2026-05|     Beauty|     68176.87|
|2025-07|     Sports|     65586.02|
|2023-04|     Sports|     49394.26|
|2026-06|      Books|      6398.72|
|2023-02|      Books|      9987.27|
+-------+-----------+-------------+
only showing top 20 rows



In [46]:
window_spec = Window.partitionBy('month').orderBy(desc("total_revenue"))

In [47]:
top_category_by_month = monthly_sales.withColumn(
        'rank',
        row_number().over(window_spec)
    ).filter(col("rank") == 1)

In [54]:
yearly_revenue = df.groupBy("category",year("order_month").alias("year")).agg(
    count("*").alias("quantity")
)

+-----------+----+--------+
|   category|year|quantity|
+-----------+----+--------+
|Electronics|2023|     175|
|      Books|2025|     237|
|     Sports|2025|     208|
|       Home|2023|     191|
|    Fashion|2026|     104|
|    Fashion|2023|     190|
|     Sports|2026|      87|
|       Home|2024|     207|
|     Beauty|2024|     246|
|      Books|2023|     182|
|Electronics|2026|      83|
|     Beauty|2026|     102|
|     Beauty|2025|     226|
|       Home|2026|     104|
|    Fashion|2024|     211|
|Electronics|2025|     200|
|      Books|2026|      77|
|     Beauty|2023|     177|
|    Fashion|2025|     222|
|      Books|2024|     228|
+-----------+----+--------+
only showing top 20 rows



In [56]:
window_spect = Window.partitionBy("year").orderBy(col("quantity").desc())

In [60]:
top_category_quantity_by_year = yearly_revenue.withColumn(
    'rank',
    row_number().over(window_spect)
).filter(col("rank") == 1)
top_category_quantity_by_year.orderBy(col("quantity").desc()).show()

+--------+----+--------+----+
|category|year|quantity|rank|
+--------+----+--------+----+
|  Beauty|2024|     246|   1|
|   Books|2025|     237|   1|
|  Sports|2023|     212|   1|
| Fashion|2026|     104|   1|
+--------+----+--------+----+

